# HRAI API demo demo notebook


In [ ]:
!pip install --upgrade pip

In [ ]:
!pip install -r requirements.txt

In [ ]:
import os, json, random, textwrap
from pathlib import Path
from fastapi.testclient import TestClient
from main import app, get_encoder

In [ ]:
# pro rychlejší stáhnutí encoderu (za předpokladu že není použitý git lfs) doporučuji nastavit HF_TOKEN:
os.environ['HF_TOKEN'] = ''

## definice funkcí

In [ ]:
client = TestClient(app)
encoder = get_encoder()  # pro encoderu do paměti; defaultní nastavení encoderu se nachází v souboru datamodels/config.py
DATA_PATH = Path('resumes_cs.json') # součást původního datasetu, nepoužito při trénování nikterého z modelů

def pick_resume():
    resumes = json.loads(DATA_PATH.read_text(encoding='utf-8'))
    resume = random.choice(resumes)
    return resume.strip()


def call_resume_endpoint(payload: dict) -> dict:
    resp = client.post('/resume', json=payload)
    resp.raise_for_status()
    return resp.json()

def call_post_entities(entities: list[dict], top_n: int = 5, extra_skill_limit: int = 10):
    payload = {
        'entities': entities,
        'top_n': top_n,
        'extra_skill_limit': extra_skill_limit,
    }
    resp = client.post('/post', json=payload)
    resp.raise_for_status()
    return resp.json()

def print_lookup_data(data: dict, max_skills: int = 8, max_domains: int = 3):
    skills = data.get('Schopnosti a dovednosti:', [])
    print(f"(zobrazeno {max_skills} z nalezených {len(skills)}):")
    for skill in skills[:max_skills]:
        print(f"  - {skill['label']} ( skóre podobnosti: {skill['score']:.3f}, zdroj: {skill['source']}, text: {skill['source_text']})")
    domains = data.get('domain_recommendations', [])
    print()
    print(f"nejlépe skórující povolání z různých oborů ESCO: (top {min(max_domains, len(domains))}/{len(domains)}):")
    for dom in domains[:max_domains]:
        occ = dom['occupation']
        print(f"  - ID oboru {dom['domain_code']} | {occ['label']} (ISCO {occ['isco_code']}, skóre (ranking) {occ['score']:.3f})")
        for extra in dom.get('extra_skills', [])[:5]:
            print(f"(možná) chybějící dovednost: \n    {extra['label']}    důležitost: ({extra['relation_type']}\n skóre (ranking): {extra['score']:.3f})")
    if data.get('occupation_suggestions'):
        print('\nESCO kategorie pro specifikovanou pozici:')
        for s in data['occupation_suggestions']:
            occ = s['occupation']
            print(f"  - {occ['label']} (skóre podobnosti: {occ['score']:.3f})")

## Testování

In [ ]:
text = pick_resume()
print(textwrap.shorten(text, width=600, placeholder='...'))

### hledání N-gramů z CV přímo v indexu
- parsování modelem Czech-pdt or ÚFAL

#### pozice + CV (ngram pipeline)
-> ESCO pozice nejbližší ke vztupu + doporučené dovednosti které nejsou blízké těm, které v CV jsou

In [ ]:
job_title = input('Sem může uživatel napsat třeba svojí "práci snů": ')
payload = {
    'resume_text': text,
    'job_title': job_title,
    'top_n': 10,
}
resp_data = call_resume_endpoint(payload)
print_lookup_data(resp_data)

#### CV input
-> Pokusí se odhadnout oblast uživatele (např. životopis na brigádu v kuchyni), a na základě toho doporučit relevantní dovednosti

In [ ]:
payload = {
    'resume_text': text,
    'job_title': None,
    'top_n': 10,
}
resp_data = call_resume_endpoint(payload)
print_lookup_data(resp_data)

### Manuální vstup

In [ ]:
manual_skills = input('vlož seznam dovedností')  # replace with any skill list
payload = {
    'resume_text': None,
    'skills': manual_skills,
    'job_title': None,
    'top_n': 10,
}
resp_data = call_resume_endpoint(payload)
print_lookup_data(resp_data)

## Dotazy

In [ ]:
def query_single_entity(text: str, entity_type: str, top_n: int = 5):
    entities = [{'text': text, 'entity_type': entity_type}]
    results = call_post_entities(entities, top_n=top_n)
    print(f"Top matches in '{entity_type}' index for '{text}':")
    for match in results:
        occ = match['occupation']
        print(f"  - {occ['label']} (ISCO {occ['isco_code']}, score {occ['score']:.3f})")
    return results

### Práce

In [ ]:
q = input()
query_single_entity(q, entity_type='occupation', top_n=5)

### Schopnost/Dovednost

In [ ]:
q = input()
query_single_entity(q, entity_type='skill', top_n=5)

### Prace - obecnější kategorie

In [ ]:
q = input()
query_single_entity(q, entity_type='isco_group', top_n=5)

### Schopnost/Dovednost - obecnější kategorie

In [ ]:
q = input()
query_single_entity(q, entity_type='skill_group', top_n=5)